In [ ]:
# Lab type: debug
# Course: DS203 — Feature Engineering & Pipelines
# Lesson: Understanding Data Leakage
# Task: The code below contains 3 bugs. Each runs without error but produces a silent
#       mistake that inflates evaluation scores or includes illegitimate information.
#       Find each bug, explain it in the markdown cell below it, and fix it.

# Lab: Debugging a Leaky Pipeline

This lab presents three standalone code blocks, each containing one data leakage bug
from the catalogue in Lesson 1. All three blocks run without Python errors —
the bugs are conceptual, not syntactic.

For each bug:
1. Run the cell to see the symptom.
2. Identify the line that introduces leakage.
3. Write your diagnosis in the markdown cell below.
4. Rewrite the fixed version in the fix cell.

**Outputs are cleared.** Run each cell to generate results.

## Setup: install dependencies and build the dataset

In [ ]:
!pip install scikit-learn pandas numpy --quiet

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

np.random.seed(42)
n = 800

# Core behavioural features
tenure_days    = np.random.exponential(300, n).clip(30, 1000).astype(int)
monthly_spend  = np.random.lognormal(4.5, 0.6, n).round(2)
support_tickets = np.random.poisson(2, n).clip(0, 15).astype(int)
plan_encoded   = np.random.choice([0, 1, 2], n, p=[0.4, 0.4, 0.2])

# True churn probability (logistic DGP)
logit = -0.003 * tenure_days + 0.4 * support_tickets - 0.02 * monthly_spend + 0.5 * plan_encoded
churn_prob = 1 / (1 + np.exp(-logit))
churned = (np.random.rand(n) < churn_prob).astype(int)

# Bug 2 helper — cancellation_flag exists only once a customer has started cancelling
cancellation_flag = churned.copy()   # identical to the target at training time

# Bug 3 helper — monthly spend history across 12 months
# We are predicting churn at month 3; months 4–12 are future data at prediction time
monthly_history = np.random.lognormal(4.5, 0.6, (n, 12))
# Churning customers reduce spend in later months (this is the signal that leaks)
monthly_history[churned == 1, 6:] *= 0.55

full_year_avg_spend = monthly_history.mean(axis=1).round(2)     # uses months 1–12 (Bug 3)
early_avg_spend     = monthly_history[:, :3].mean(axis=1).round(2)  # only months 1–3 (correct)

df = pd.DataFrame({
    "tenure_days":        tenure_days,
    "monthly_spend":      monthly_spend,
    "support_tickets":    support_tickets,
    "plan_encoded":       plan_encoded,
    "cancellation_flag":  cancellation_flag,
    "full_year_avg_spend": full_year_avg_spend,
    "early_avg_spend":    early_avg_spend,
    "churned":            churned,
})

print(f"Dataset: {df.shape[0]} customers, churn rate {df['churned'].mean():.1%}")
df.head()

## Bug 1: Fit-on-all

The preprocessing step below runs without error and produces a plausible AUC.
Compare the result to the reference honest evaluation printed alongside it.

In [ ]:
# Reference — honest evaluation (split before fitting the scaler)
X_raw = df[["tenure_days", "monthly_spend", "support_tickets", "plan_encoded"]]
y = df["churned"]

X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y, test_size=0.2, random_state=42)
scaler_ref = StandardScaler()
X_tr_s = scaler_ref.fit_transform(X_tr)
X_te_s = scaler_ref.transform(X_te)
model_ref = LogisticRegression(max_iter=500)
model_ref.fit(X_tr_s, y_tr)
auc_ref = roc_auc_score(y_te, model_ref.predict_proba(X_te_s)[:, 1])
print(f"AUC (honest):  {auc_ref:.3f}")

# --- BUGGY CODE (Bug 1) ---
# Review this code — is it correct?
X = df[["tenure_days", "monthly_spend", "support_tickets", "plan_encoded"]]
y = df["churned"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)          # ← fit on all data, including test rows

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
model_bug1 = LogisticRegression(max_iter=500)
model_bug1.fit(X_train, y_train)
auc_bug1 = roc_auc_score(y_test, model_bug1.predict_proba(X_test)[:, 1])
print(f"AUC (bug 1):   {auc_bug1:.3f}")
print(f"Inflated by:   {auc_bug1 - auc_ref:+.3f}")

**Explain the bug:** Which line introduces the leakage? What information does the test set contribute to the scaler that it shouldn’t?

*(Write your diagnosis here.)*

In [ ]:
# Fix for Bug 1: split first, then fit the scaler only on training data
X = df[["tenure_days", "monthly_spend", "support_tickets", "plan_encoded"]]
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler_fixed = StandardScaler()
X_train_s = scaler_fixed.fit_transform(X_train)  # fit on training rows only
X_test_s  = scaler_fixed.transform(X_test)       # transform only (no fit)

model_fix1 = LogisticRegression(max_iter=500)
model_fix1.fit(X_train_s, y_train)
auc_fix1 = roc_auc_score(y_test, model_fix1.predict_proba(X_test_s)[:, 1])
print(f"AUC (fix 1):   {auc_fix1:.3f}")
print(f"Matches honest: {abs(auc_fix1 - auc_ref) < 1e-9}")

## Bug 2: Target leakage

The feature set below includes a column that wouldn’t exist at prediction time
for a live customer. The model appears highly accurate because it has essentially
been given the answer.

In [ ]:
# --- BUGGY CODE (Bug 2) ---
# Review this feature set — does every column pass the availability check?
feature_cols_bug2 = [
    "tenure_days",
    "monthly_spend",
    "support_tickets",
    "plan_encoded",
    "cancellation_flag",   # ← does this exist at prediction time for a live customer?
]

X_bug2 = df[feature_cols_bug2]
y = df["churned"]

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_bug2, y, test_size=0.2, random_state=42)
pipeline_bug2 = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(max_iter=500)),
])
pipeline_bug2.fit(X_tr2, y_tr2)
auc_bug2 = roc_auc_score(y_te2, pipeline_bug2.predict_proba(X_te2)[:, 1])
print(f"AUC with cancellation_flag: {auc_bug2:.3f}")

# Inspect the relationship between the flag and the target
print("\ncancellation_flag vs churned:")
print(df.groupby("cancellation_flag")["churned"].value_counts())
print(f"\nCorrelation: {df['cancellation_flag'].corr(df['churned']):.3f}")

**Explain the bug:** Why is `cancellation_flag` a leakage feature? Would this column exist in production at the moment you make a churn prediction for a live customer?

*(Write your diagnosis here.)*

In [ ]:
# Fix for Bug 2: remove the target-leakage feature from the feature set
feature_cols_fix2 = [
    "tenure_days",
    "monthly_spend",
    "support_tickets",
    "plan_encoded",
    # cancellation_flag removed — not available at prediction time
]

X_fix2 = df[feature_cols_fix2]
y = df["churned"]

X_tr2f, X_te2f, y_tr2f, y_te2f = train_test_split(X_fix2, y, test_size=0.2, random_state=42)
pipeline_fix2 = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(max_iter=500)),
])
pipeline_fix2.fit(X_tr2f, y_tr2f)
auc_fix2 = roc_auc_score(y_te2f, pipeline_fix2.predict_proba(X_te2f)[:, 1])
print(f"AUC without cancellation_flag: {auc_fix2:.3f}")
print(f"AUC with cancellation_flag:    {auc_bug2:.3f}")
print(f"Gap caused by target leakage:  {auc_bug2 - auc_fix2:+.3f}")

## Bug 3: Temporal leakage

We are predicting churn at **month 3** of a customer’s lifecycle.
One feature below was engineered from data that only becomes available after the prediction point.

In [ ]:
# --- BUGGY CODE (Bug 3) ---
# Review this feature set — which column uses future data?
feature_cols_bug3 = [
    "tenure_days",
    "support_tickets",
    "plan_encoded",
    "full_year_avg_spend",  # ← average of months 1–12; prediction is at month 3
]

X_bug3 = df[feature_cols_bug3]
y = df["churned"]

X_tr3, X_te3, y_tr3, y_te3 = train_test_split(X_bug3, y, test_size=0.2, random_state=42)
pipeline_bug3 = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(max_iter=500)),
])
pipeline_bug3.fit(X_tr3, y_tr3)
auc_bug3 = roc_auc_score(y_te3, pipeline_bug3.predict_proba(X_te3)[:, 1])
print(f"AUC with full_year_avg_spend (temporal leak): {auc_bug3:.3f}")

# Show the hidden signal: churners reduce spend after the prediction point
print("\nMean full_year_avg_spend by churn outcome:")
print(df.groupby("churned")[["full_year_avg_spend", "early_avg_spend"]].mean().round(2))
print()
print("full_year_avg_spend — correlation with churned:", df["full_year_avg_spend"].corr(df["churned"]).round(3))
print("early_avg_spend     — correlation with churned:", df["early_avg_spend"].corr(df["churned"]).round(3))

**Explain the bug:** We are predicting churn at month 3. `full_year_avg_spend` averages months 1–12. What months are unavailable at the prediction point? Why does this inflate the model’s apparent accuracy?

*(Write your diagnosis here.)*

In [ ]:
# Fix for Bug 3: replace full_year_avg_spend with early_avg_spend (months 1–3 only)
feature_cols_fix3 = [
    "tenure_days",
    "support_tickets",
    "plan_encoded",
    "early_avg_spend",  # only months 1–3 — available at prediction time
]

X_fix3 = df[feature_cols_fix3]
y = df["churned"]

X_tr3f, X_te3f, y_tr3f, y_te3f = train_test_split(X_fix3, y, test_size=0.2, random_state=42)
pipeline_fix3 = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(max_iter=500)),
])
pipeline_fix3.fit(X_tr3f, y_tr3f)
auc_fix3 = roc_auc_score(y_te3f, pipeline_fix3.predict_proba(X_te3f)[:, 1])
print(f"AUC with early_avg_spend (honest):       {auc_fix3:.3f}")
print(f"AUC with full_year_avg_spend (leaked):   {auc_bug3:.3f}")
print(f"Gap caused by temporal leakage:          {auc_bug3 - auc_fix3:+.3f}")

## Summary

> **For each bug, complete this sentence in one line.**

1. **Fit-on-all:** The scaler was fitted on _______, which means the test set contributed _______ to the scaler before evaluation.
2. **Target leakage:** `cancellation_flag` is a leak because _______.
3. **Temporal leakage:** `full_year_avg_spend` is a leak because at prediction time (month 3), months _______ have not yet occurred.